In [1]:
from pathlib import Path
TRITON_API_KEY = Path('../api-key.txt').expanduser().read_text(encoding='utf-8').splitlines()[0].strip()
BASE_URL = 'https://tritonai-api.ucsd.edu'

In [2]:
from typing import Dict, Any
from typing import List as listtype
import re
from collections import defaultdict
import pandas as pd

In [3]:
# document loaders
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_core.documents import Document

# text splitter
from langchain_text_splitters import RecursiveCharacterTextSplitter

# embeddings
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import OpenAIEmbeddings

# retriever
from langchain_core.retrievers import BaseRetriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_community.retrievers import TFIDFRetriever, AmazonKnowledgeBasesRetriever

# cross encoder for reranking
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker

In [4]:
# rapidfire imports
from rapidfireai.automl import (
    List,
    RFLangChainRagSpec,
    RFOpenAIAPIModelConfig,
    RFPromptManager,
    RFGridSearch,
)
from rapidfireai import Experiment

In [5]:
from rapidfire_integration_example import sample_compute_metrics_fn, sample_accumulate_metrics_fn
from project1_eval import to_spans

In [6]:
import os

os.environ['OPENAI_API_KEY'] = TRITON_API_KEY
os.environ['JUDGE_BASE_URL'] = BASE_URL
os.environ['JUDGE_MODEL'] = 'api-mistral-small-3.2-2506'

[https://oss-docs.rapidfire.ai/en/latest/ragspecs.html](https://oss-docs.rapidfire.ai/en/latest/ragspecs.html)

In [7]:
class TextLoaderWithMetadata(TextLoader):
    def __init__(self, file_path: str, metadata_func=None, **kwargs):
        super().__init__(file_path=file_path, **kwargs)
        self.metadata_func = metadata_func
    
    def load(self) -> list[Document]:
        # load the documents as TextLoader
        docs = super.load()
        # if a metadata function is specified
        if self.metadata_func is not None:
            for d in docs:
                d.metadata = self.metadata_func(d, d.metadata)
        
        return docs
    
    def lazy_load(self):
        # for each document loaded using TextLoader
        for doc in super().lazy_load():
            # if a metadata function is specified
            if self.metadata_func is not None:
                # apply the metadata function to the document
                doc.metadata = self.metadata_func(doc, dict(doc.metadata))
            yield doc

In [8]:
def metadata_func(document, metadata):
    metadata['source'] = Path(metadata['source']).name
    return metadata

In [9]:
loader = DirectoryLoader(
    path='../data/sourcedocs/',
    glob='**/*.rst',
    loader_cls=TextLoaderWithMetadata,
    loader_kwargs={
        'metadata_func': metadata_func,
    },
    show_progress=True,
    use_multithreading=True
)

Models:
- `gpt2` / `r50k_base`: older ~50k-vocab GPT-2 style tokenizers. `r50k_base` is the standard GPT-2 BPE; `gpt2` is the GPT-2 encoding entry in `tiktoken`. Both use the same regex pattern and the same end-of-text token, with `gpt2` built from the original GPT-2 encoder/vocab files
- `p50k_base`: also ~50k vocab, but different merge ranks from `r50k_base`; intended for Codex-style/code-heavy tokenization
- `p50k_edit`: same base merges as `p50k_base`, but adds extra FIM/edit special tokens like `<|fim_prefix|>`, `<|fim_middle|>`, and `<|fim_suffix|>`
- `cl100k_base`: newer ~100k-vocab tokenizer with a different regex and a larger special-token set; commonly associated with GPT-4 / GPT-3.5-era model families
- `o200k_base`: newer ~200k-vocab tokenizer with a different regex again; used by newer model families such as `gpt-4o`
- `o200k_harmony`: derived from `o200k_base` but with many additional special tokens layered on top

In [10]:
class TextSplitterWithLineNum(RecursiveCharacterTextSplitter):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
    
    def split_documents(self, documents):
        split = super().split_documents(documents)

        # create a mapping for every document
        line_num_map = {}
        for d in documents:
            file_name = d.metadata['source']
            m = defaultdict(list)
            # iterate through each line and give it a mapping
            for idx, line in enumerate(d.page_content.split('\n')):
                m[line.strip()].append(idx + 1)
            line_num_map[file_name] = m

        # for each chunk find the line index mapping for the first and last line
        for chunk in split:
            source = chunk.metadata['source']
            lines = chunk.page_content.strip().split('\n')
            # get the first and last line of the chunk
            first, last = line_num_map[source][lines[0].strip()], line_num_map[source][lines[-1].strip()]

            # find the start index and 
            start = first.pop(0)
            # choose the first matching end line at or after start
            end = next(i for i in last if i >= start)

            chunk.metadata['source'] = {"file": source, 'lines': [start, end]}
        return split

In [ ]:
text_splitter = TextSplitterWithLineNum().from_tiktoken_encoder(
    encoding_name='o200k_base',
    chunk_size=512, 
    chunk_overlap=64,
    separators=['\n', '\n\n']
)

From [RapidFire AI oss](https://oss-docs.rapidfire.ai/en/latest/ragspecs.html)

`embedding_cfg (dict[str, Any], optional)`

The embedding class and its kwargs to convert a chunk/query into a vector, provided as a single dictionary. Must include a key `"class"` with the class itself as value, not an instance. Options for the class include `HuggingFaceEmbeddings` and `OpenAIEmbeddings`. 

The kwargs that follow must contain all parameters needed to initialize the embedding class; required parameters vary by embedding class. For example, `HuggingFaceEmbeddings` needs `model_name`, `model_kwargs` and `device`, while `OpenAIEmbeddings` needs `"model"` and `"api_key"`.

In [12]:
batch_size = 32

embedding_cfg = List([
    # yet to use open ai embeddings
    {
        'class': HuggingFaceEmbeddings,
        'model_name': 'sentence-transformers/all-MiniLM-L6-v2',
        'model_kwargs': {'device': 'cpu'},
        'encode_kwargs': {'normalize_embeddings': True, 'batch_size': batch_size}
    },
    {
        'class': OpenAIEmbeddings,
        'model': 'api-tgpt-embeddings',
        'api_key': TRITON_API_KEY,
        'base_url': BASE_URL,
        'check_embedding_ctx_length': False
    }
])

From [RapidFire AI oss](https://oss-docs.rapidfire.ai/en/latest/ragspecs.html)

`vector_store_cfg (dict[str, Any], optional)`

The vector store type and args to store and possibly index embedding vectors for retrieval, provided as a single dictionary.

- `"type"`: The type of vector store to use. Must be one of `"faiss"`, `"pgvector"`, or `"pinecone"`. Required.
- `"batch_size"`: Number of vectors per insert batch. Applies to all 3 types of stores. Optional; default is 128.

In [13]:
vector_store_cfg={
    'type': 'faiss', 
    'batch_size': batch_size
}

In [ ]:
search_cfg=List([
    {'type': 'similarity', 'k': 10},
    {'type': 'mmr', 'k': 10}
])

In [ ]:
reranker_cfg=List([
    {
        'class': CrossEncoderReranker,
        'model_name': 'cross-encoder/ms-marco-MiniLM-L6-v2',
        'model_kwargs': {'device': 'cpu'},
        'top_n': 5,
    },
    {
        'class': CrossEncoderReranker,
        'model_name': 'cross-encoder/ms-marco-MiniLM-L12-v2',
        'model_kwargs': {'device': 'cpu'},
        'top_n': 5,
    }
])

In [16]:
def custom_template(doc):
    return f'{doc.metadata["source"]}: {doc.page_content}'

In [17]:
rag = RFLangChainRagSpec(
    # to load the corpus
    document_loader=loader,
    # to split documents into chunks
    text_splitter=text_splitter,
    # to embed the chunks
    embedding_cfg=embedding_cfg,
    # vectore storage
    vector_store_cfg=vector_store_cfg,
    # search algorithm
    search_cfg=search_cfg,
    # rerank top results with a cross encoder
    reranker_cfg=reranker_cfg,
    # whether to use gpu-accelerated faiss
    enable_gpu_search=False,
    # to format each retrieved chunk for context injection into prompts
    document_template=custom_template,
)

In [18]:
INSTRUCTIONS = """
You are a helpful assistant answering questions about RapidFire AI docuentation.

Your job:
- Read the retrieved documentation context
- Answer the user's question using only the context
- Do not make up information that is not supported by the retrieved text

Rules:
1. Use only the retrieved document context.
2. Keep the answer concise but complete.
3. The answer should be in plain text and do not use any markdown syntax including list.
4. Mention relevant file names and the corresponding start and end line number when they are clearly useful.
5. There can be multiple sources but only keep the ones that are really relevant.
6. If there are sources from the same file, only keep the most relevant one.

Output format:
#### ANSWER: <final answer>
#### SOURCE: [{"file": <source file>, "lines": [<start line>, <end line>]}]
"""

In [19]:
import json
from datasets import Dataset

In [20]:
def load_eval_question_set(path, encoding='utf-8', validation=False):
    # load the questions from the file
    with open(path, 'r', encoding=encoding) as f:
        eval_questions = json.load(f)
    
    # to store each example
    eval_rows = []

    for example in eval_questions:
        # extract the question and id
        row = {
            'question_id': example['question_id'],
            'question': example['question']
        }
        # validation sets include reference answer an source evidence
        if validation:
            row['reference_answer'] = example['reference_answer']
            row['source_evidence'] = example['source_evidence']
        
        eval_rows.append(row)

    
    # convert into a hugging face dataset
    eval_dataset = Dataset.from_list(eval_rows)
    return eval_dataset

golden_qa_val = load_eval_question_set(path='../data/validation-set/validation-set-golden-qa-pairs.json', validation=True)

In [21]:
def sample_preprocess_fn(batch: Dict[str, listtype], 
                         rag: RFLangChainRagSpec, 
                         prompt_manager: RFPromptManager) -> Dict[str, listtype]:
    '''
    convert a batch of dataset rows into model-ready prompts
    '''
    # retrieve context for each question in the batch
    all_context = rag.get_context(batch_queries=batch['question'], serialize=False)
    # extract the source metadata field and retrieved context from each retrieved document
    retrieved_documents, retrieved_contexts = [], []
    for docs in all_context:
        evidence_for_query, context = [], []
        for doc in docs:
            evidence_for_query.append(doc.metadata.get('source'))
            context.append(doc.page_content)
        # attach the source evidence and retrieved context to the question
        retrieved_documents.append(evidence_for_query)
        retrieved_contexts.append(context)

    # convert retrieved docs into plain text strings suitable for prompt injection
    serialized_context = rag.serialize_documents(all_context)

    # build a chat-style prompt per question
    prompts = [
        [
            # instruction as system prompt
            {'role': 'system', 'content': INSTRUCTIONS},
            # user message containing the question
            {
                'role': 'user', 'content': f'Question: {question} \nDocumentation context: {context} \nYour response:'
            }
        ]
        for question, context in zip(batch['question'], serialized_context)
    ]

    return {
        'prompts': prompts,
        'retrieved_documents': retrieved_documents,
        'retrieved_contexts': retrieved_contexts,
        'serialized_context': serialized_context,
        **batch
    }

In [22]:
def extract_solution(answer: str) -> str:
    '''
    extract the answer
    '''
    # search for the answer marker
    answer_match = re.search(
        r"####\s*Answer:\s*(.*?)(?=\n####\s*Source:|\Z)",
        answer,
        flags=re.IGNORECASE | re.DOTALL,
    )
    # search for the source marker
    source_match = re.search(
        r"####\s*Source:\s*(.*)",
        answer,
        flags=re.IGNORECASE | re.DOTALL,
    )
    answer = answer_match.group(1).strip() if answer_match else answer
    source = source_match.group(1).strip() if source_match else None
    return answer, source

In [ ]:
def sample_postprocess_fn(batch: Dict[str, listtype]) -> Dict[str, listtype]:
    '''
    turn raw generated text into a field
    '''
    # extract answer and source from the generated text
    answer, source = [], []
    for text in batch['generated_text']:
        a, s = extract_solution(text)
        answer.append(a)
        source.append(s)
    batch['answer'] = answer
    batch['source'] = source
    # just different naming for the metrics
    parsed_sources = []
    for s in batch['source']:
        if s is None:
            parsed_sources.append([])
        else:
            try: 
                parsed_sources.append(json.loads(s))
            except json.JSONDecodeError:
                parsed_sources.append([])
    batch['retrieved_spans'] = [to_spans(s) for s in parsed_sources]
    batch['ground_truth_spans'] = [to_spans(s) for s in batch['source_evidence']]
    batch['query'] = batch['question']
    return batch

In [ ]:
openai_config1 = RFOpenAIAPIModelConfig(
    client_config={'api_key': TRITON_API_KEY, 'base_url': BASE_URL, 'max_retries': 2},
    model_config={
        'model': 'api-mistral-small-3.2-2506',
        'max_completion_tokens': 2048,
        # two different reasoning levels
        'reasoning_effort': 'high'
    },
    # request per minute needs to be set based on your account tier and the specific model used
    rpm_limit=120,
    # token per minute needs to be set based on your account tier and the specific model used
    tpm_limit=1_000_000,
    # rag model
    rag=rag,
    prompt_manager=None
)
openai_config2 = RFOpenAIAPIModelConfig(
    client_config={'api_key': TRITON_API_KEY, 'base_url': BASE_URL, 'max_retries': 2},
    model_config={
        'model': 'api-gpt-oss-120b',
        'max_completion_tokens': 2048,
        # two different reasoning levels
        'reasoning_effort': 'high'
    },
    # request per minute needs to be set based on your account tier and the specific model used
    rpm_limit=120,
    # token per minute needs to be set based on your account tier and the specific model used
    tpm_limit=1_000_000,
    # rag model
    rag=rag,
    prompt_manager=None
)

In [25]:
config_set = List([{
    'openai_config': List([
        openai_config1, openai_config2
    ]),
    'batch_size': batch_size,
    'preprocess_fn': sample_preprocess_fn,
    'postprocess_fn': sample_postprocess_fn,
    # add metrics later
    'compute_metrics_fn': sample_compute_metrics_fn,
    'accumulate_metrics_fn': sample_accumulate_metrics_fn,
    'online_strategy_kwargs': {
        'strategy_name': 'normal',
        'confidence_level': 0.95,
        'use_fpc': True,
    }
}])

config_group = RFGridSearch(config_set)

In [26]:
# create experiment
experiment = Experiment(
    experiment_name='rag-generation-only',
    mode='evals'
)

# run generation
results = experiment.run_evals(
    config_group=config_group,
    dataset=golden_qa_val,
    num_shards=4,
    seed=42
)
experiment.end()

An experiment with the same name already exists. Created a new experiment 'rag-generation-only_53' with Experiment ID: 55 at /Users/chenchiahan/rapidfireai/rapidfire_experiments/rag-generation-only_53
Created directory: /Users/chenchiahan/rapidfireai/logs/rag-generation-only_53


=== Preprocessing RAG Sources ===


RAG Source ID,Status,Duration,Device,Vector Store
1,Complete,15.3s,CPU,FAISS
2,Complete,16.4s,CPU,FAISS
3,Complete,16.4s,CPU,FAISS
4,Complete,16.4s,CPU,FAISS



=== Multi-Config Experiment Progress ===


Run ID,Model,Status,Progress,Conf. Interval,text_splitter,chunk_size,chunk_overlap,embedding_cfg.base_url,embedding_cfg.check_embedding_ctx_length,embedding_cfg.class,embedding_cfg.encode_kwargs.batch_size,embedding_cfg.encode_kwargs.normalize_embeddings,embedding_cfg.model,embedding_cfg.model_name,vector_store_cfg.batch_size,vector_store_cfg.type,search_cfg.fetch_k,search_cfg.k,search_cfg.lambda_mult,search_cfg.type,reranker_cfg.class,reranker_cfg.model_name,reranker_cfg.top_n,model_config,Completeness (normalized),Correctness (pass rate),F1@5,Faithfulness (pass rate),Generation Score (3 released),Judge Failures,Precision@5,Processing Time,Recall@5,Retrieval Score,Samples Per Second,Samples Processed,Throughput,Total
1,api-mistral-small-3.2-2506,COMPLETED,4/4,0.000,TextSplitterWithLineNum,512,64,-,-,HuggingFaceEmbeddings,32,True,-,sentence-transformers/all-MiniLM-L6-v2,32,faiss,-,10,-,similarity,CrossEncoderReranker,cross-encoder/ms-marco-MiniLM-L6-v2,5,"max_completion_tokens=2048, reasoning_effort=high","0.7689 [0.7689, 0.7689]","0.8667 [0.8667, 0.8667]","0.2568 [0.2568, 0.2568]","0.9778 [0.9778, 0.9778]","0.8711 [0.8711, 0.8711]",0.0000,"0.1600 [0.1600, 0.1600]",3074.85 seconds,"0.7056 [0.7056, 0.7056]","0.3741 [0.3741, 0.3741]",0.01,45,0.0/s,45
2,api-mistral-small-3.2-2506,COMPLETED,4/4,0.000,TextSplitterWithLineNum,512,64,-,-,HuggingFaceEmbeddings,32,True,-,sentence-transformers/all-MiniLM-L6-v2,32,faiss,20,10,0.5,mmr,CrossEncoderReranker,cross-encoder/ms-marco-MiniLM-L6-v2,5,"max_completion_tokens=2048, reasoning_effort=high","0.7111 [0.7111, 0.7111]","0.8000 [0.8000, 0.8000]","0.2515 [0.2515, 0.2515]","0.9556 [0.9556, 0.9556]","0.8222 [0.8222, 0.8222]",0.0000,"0.1556 [0.1556, 0.1556]",3042.22 seconds,"0.7056 [0.7056, 0.7056]","0.3709 [0.3709, 0.3709]",0.01,45,0.0/s,45
3,api-mistral-small-3.2-2506,COMPLETED,4/4,0.000,TextSplitterWithLineNum,512,64,https://tritonai-api.ucsd.edu,False,OpenAIEmbeddings,-,-,api-tgpt-embeddings,-,32,faiss,-,10,-,similarity,CrossEncoderReranker,cross-encoder/ms-marco-MiniLM-L6-v2,5,"max_completion_tokens=2048, reasoning_effort=high","0.7689 [0.7689, 0.7689]","0.9333 [0.9333, 0.9333]","0.2864 [0.2864, 0.2864]","0.9556 [0.9556, 0.9556]","0.8859 [0.8859, 0.8859]",0.0000,"0.1778 [0.1778, 0.1778]",3015.97 seconds,"0.7944 [0.7944, 0.7944]","0.4195 [0.4195, 0.4195]",0.01,45,0.0/s,45
4,api-mistral-small-3.2-2506,COMPLETED,4/4,0.000,TextSplitterWithLineNum,512,64,https://tritonai-api.ucsd.edu,False,OpenAIEmbeddings,-,-,api-tgpt-embeddings,-,32,faiss,20,10,0.5,mmr,CrossEncoderReranker,cross-encoder/ms-marco-MiniLM-L6-v2,5,"max_completion_tokens=2048, reasoning_effort=high","0.7733 [0.7733, 0.7733]","0.9333 [0.9333, 0.9333]","0.3076 [0.3076, 0.3076]","1.00 [1.00, 1.00]","0.9022 [0.9022, 0.9022]",0.0000,"0.1911 [0.1911, 0.1911]",2992.18 seconds,"0.8500 [0.8500, 0.8500]","0.4496 [0.4496, 0.4496]",0.02,45,0.0/s,45
5,api-mistral-small-3.2-2506,COMPLETED,4/4,0.000,TextSplitterWithLineNum,1024,256,-,-,HuggingFaceEmbeddings,32,True,-,sentence-transformers/all-MiniLM-L6-v2,32,faiss,-,10,-,similarity,CrossEncoderReranker,cross-encoder/ms-marco-MiniLM-L6-v2,5,"max_completion_tokens=2048, reasoning_effort=high","0.7333 [0.7333, 0.7333]","0.8444 [0.8444, 0.8444]","0.2557 [0.2557, 0.2557]","0.9556 [0.9556, 0.9556]","0.8444 [0.8444, 0.8444]",0.0000,"0.1556 [0.1556, 0.1556]",2968.31 seconds,"0.7500 [0.7500, 0.7500]","0.3871 [0.3871, 0.3871]",0.02,45,0.0/s,45
6,api-mistral-small-3.2-2506,COMPLETED,4/4,0.000,TextSplitterWithLineNum,1024,256,-,-,HuggingFaceEmbeddings,32,True,-,sentence-transformers/all-MiniLM-L6-v2,32,faiss,20,10,0.5,mmr,CrossEncoderReranker,cross-encoder/ms-marco-MiniLM-L6-v2,5,"max_completion_tokens=2048, reasoning_effort=high","0.6756 [0.6756, 0.6756]","0.7556 [0.7556, 0.7556]","0.2557 [0.2557, 0.2557]","0.9778 [0.9778, 0.9778]","0.8030 [0.8030, 0.8030]",0.0000,"0.1556 [0.1556, 0.1556]",2938.33 seconds,"0.7500 [0.7500, 0.7500]","0.3871 [0.3871, 0.3871]",0.02,45,0.0/s,45
7,api-mistral-small-3.2-2506,COM


⚠️  Run 11 (Pipeline 11) FAILED
Shard: 2/4
Error: ray::QueryProcessingActor.process_batch() (pid=88174, ip=127.0.0.1, actor_id=e0104c5c7574a8f13d3ba15f01000000, repr=<rapidfireai.evals.actors.query_actor.QueryProcessingActor object at 0x3c04dc920>)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/miniconda3/lib/python3.12/site-packages/rapidfireai/evals/actors/query_actor.py", line 401, in process_batch
    raise RuntimeError(
RuntimeError: Error processing batch: JSONDecodeError: Expecting ',' delimiter: line 1 column 45 (char 44)
This run has been marked as FAILED. The experiment will continue with other runs.


⚠️  Run 21 (Pipeline 21) FAILED
Shard: 2/4
Error: ray::QueryProcessingActor.process_batch() (pid=88174, ip=127.0.0.1, actor_id=e0104c5c7574a8f13d3ba15f01000000, repr=<rapidfireai.evals.actors.query_actor.QueryProcessingActor object at 0x3c04dc920>)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
           ^^^^^^^^^^^^^^

In [ ]:
results_df = pd.DataFrame([
    {k: v['value'] if isinstance(v, dict) and 'value' in v else v for k, v in {**metrics_dict, 'run_id': run_id}.items()}
    for run_id, (_, metrics_dict) in results.items()
])

pd.set_option('display.max_colwidth', None)

results_df.to_csv('experiments.csv', index=False)

results_df

,run_id,model_name,text_splitter,chunk_size,chunk_overlap,embedding_cfg,vector_store_cfg,search_cfg,reranker_cfg,model_config,Samples Processed,Processing Time,Samples Per Second,Total,F1@5,Precision@5,Recall@5,Retrieval Score,Correctness (pass rate),Faithfulness (pass rate),Completeness (normalized),Generation Score (3 released),Judge Failures
0,1,api-mistral-small-3.2-2506,TextSplitterWithLineNum,512,64,"{'class': 'HuggingFaceEmbeddings', 'model_name': 'sentence-transformers/all-MiniLM-L6-v2', 'model_kwargs': {'device': 'cpu'}, 'encode_kwargs': {'normalize_embeddings': True, 'batch_size': 32}}","{'type': 'faiss', 'batch_size': 32}","{'type': 'similarity', 'k': 10}","{'class': 'CrossEncoderReranker', 'model_name': 'cross-encoder/ms-marco-MiniLM-L6-v2', 'model_kwargs': {'device': 'cpu'}, 'top_n': 5}","{'max_completion_tokens': 2048, 'reasoning_effort': 'high'}",45,3074.85 seconds,0.01,45,0.256790,0.160000,0.705556,0.374115,0.866667,0.977778,0.768889,0.871111,0
1,2,api-mistral-small-3.2-2506,TextSplitterWithLineNum,512,64,"{'class': 'HuggingFaceEmbeddings', 'model_name': 'sentence-transformers/all-MiniLM-L6-v2', 'model_kwargs': {'device': 'cpu'}, 'encode_kwargs': {'normalize_embeddings': True, 'batch_size': 32}}","{'type': 'faiss', 'batch_size': 32}","{'type': 'mmr', 'k': 10, 'fetch_k': 20, 'lambda_mult': 0.5}","{'class': 'CrossEncoderReranker', 'model_name': 'cross-encoder/ms-marco-MiniLM-L6-v2', 'model_kwargs': {'device': 'cpu'}, 'top_n': 5}","{'max_completion_tokens': 2048, 'reasoning_effort': 'high'}",45,3042.22 seconds,0.01,45,0.251499,0.155556,0.705556,0.370870,0.800000,0.955556,0.711111,0.822222,0
2,3,api-mistral-small-3.2-2506,TextSplitterWithLineNum,512,64,"{'class': 'OpenAIEmbeddings', 'model': 'api-tgpt-embeddings', 'api_key': 'sk-YUwRq2iW87kSBXBTJN8SFw', 'base_url': 'https://tritonai-api.ucsd.edu', 'check_embedding_ctx_length': False}","{'type': 'faiss', 'batch_size': 32}","{'type': 'similarity', 'k': 10}","{'class': 'CrossEncoderReranker', 'model_name': 'cross-encoder/ms-marco-MiniLM-L6-v2', 'model_kwargs': {'device': 'cpu'}, 'top_n': 5}","{'max_completion_tokens': 2048, 'reasoning_effort': 'high'}",45,3015.97 seconds,0.01,45,0.286420,0.177778,0.794444,0.419547,0.933333,0.955556,0.768889,0.885926,0
3,4,api-mistral-small-3.2-2506,TextSplitterWithLineNum,512,64,"{'class': 'OpenAIEmbeddings', 'model': 'api-tgpt-embeddings', 'api_key': 'sk-YUwRq2iW87kSBXBTJN8SFw', 'base_url': 'https://tritonai-api.ucsd.edu', 'check_embedding_ctx_length': False}","{'type': 'faiss', 'batch_size': 32}","{'type': 'mmr', 'k': 10, 'fetch_k': 20, 'lambda_mult': 0.5}","{'class': 'CrossEncoderReranker', 'model_name': 'cross-encoder/ms-marco-MiniLM-L6-v2', 'model_kwargs': {'device': 'cpu'}, 'top_n': 5}","{'max_completion_tokens': 2048, 'reasoning_effort': 'high'}",45,2992.18 seconds,0.02,45,0.307584,0.191111,0.850000,0.449565,0.933333,1.000000,0.773333,0.902222,0
4,5,api-mistral-small-3.2-2506,TextSplitterWithLineNum,1024,256,"{'class': 'HuggingFaceEmbeddings', 'model_name': 'sentence-transformers/all-MiniLM-L6-v2', 'model_kwargs': {'device': 'cpu'}, 'encode_kwargs': {'normalize_embeddings': True, 'batch_size': 32}}","{'type': 'faiss', 'batch_size': 32}","{'type': 'similarity', 'k': 10}","{'class': 'CrossEncoderReranker', 'model_name': 'cross-encoder/ms-marco-MiniLM-L6-v2', 'model_kwargs': {'device': 'cpu'}, 'top_n': 5}","{'max_completion_tokens': 2048, 'reasoning_effort': 'high'}",45,2968.31 seconds,0.02,45,0.255732,0.155556,0.750000,0.387096,0.844444,0.955556,0.733333,0.844444,0
5,6,api-mistral-small-3.2-2506,TextSplitterWithLineNum,1024,256,"{'class': 'HuggingFaceEmbeddings', 'model_name': 'sentence-transformers/all-MiniLM-L6-v2', 'model_kwargs': {'device': 'cpu'}, 'encode_kwargs': {'normalize_embeddings': True, 'batch_size': 32}}","{'type': 'faiss', 'batch_size': 32}","{'type': 'mmr', 'k': 10, 'fetch_k': 20, 'lambda_mult': 0.5}","{'class': 'CrossEncoderReranker', 'model_name': 'cross-encoder/ms-marco-MiniLM-L6-v2', 'model_kwargs': {'device':

: 